# Gold Layer — Fraud Analytics & Loss Prevention

This notebook documents the **declarative gold layer** implemented in
`src/transformations/gold/gold_aggregations.py` (pipeline `beestats_medallion_pipeline`).
It reads the cleaned Silver tables from `workspace.silver` and builds 4 Gold deliverables
published to `workspace.gold`:

1. `gold_employee_fraud_scorecard` — per-employee fraud flag aggregation
2. `gold_location_loss_prevention_summary` — per-location loss metrics
3. `gold_hourly_anomaly_trends` — hourly anomaly patterns by location
4. `gold_suspect_check_details` — audit dataset for receipt-level investigation

**Key characteristics:**
* All 4 deliverables are **materialized views** (batch `spark.read.table` reads) — the pipeline
  handles the joins, aggregations, and incremental refreshes automatically.
* The first 3 are **liquid-clustered on `LocationId`** (`cluster_by=["LocationId"]`), the same
  layout the legacy notebook created with `CREATE TABLE ... CLUSTER BY (LocationId)`.
* All four read from `workspace.silver.suspect_fraud_checks` (plus the `employees` /
  `revenue_centers` dimensions for readable names).

> **Note:** this notebook is **documentation only** — it is *not* part of the pipeline `libraries`.
> The pipeline source of truth is `gold_aggregations.py`; the runnable cells below simply
> query the materialized gold tables for verification and previews.


## Dataset Inventory (4 materialized views)

| Gold dataset | Source(s) | Columns | Description |
| --- | --- | --- | --- |
| `workspace.gold.gold_employee_fraud_scorecard` | `silver.suspect_fraud_checks` + `silver.employees` | `employee_id`, `employee_name`, `LocationId`, `total_suspect_checks`, `high_val_discount_flags`, `low_val_discount_flags`, `irregular_hours_flags`, `negative_price_flags`, `suspicious_togo_flags`, `total_discount_dollars_at_risk`, `total_void_dollars_at_risk` | One row per employee × location; suspect-check count, per-hypothesis flag totals, and discount/void dollars at risk. Clustered by `LocationId`. |
| `workspace.gold.gold_location_loss_prevention_summary` | `silver.suspect_fraud_checks` + `silver.revenue_centers` | `LocationId`, `revenue_center_name`, `total_flagged_checks`, `total_discount_loss`, `total_void_loss`, `total_dollars_at_risk` | One row per location × revenue center; flagged-check count and combined discount+void exposure. Clustered by `LocationId`. |
| `workspace.gold.gold_hourly_anomaly_trends` | `silver.suspect_fraud_checks` | `LocationId`, `hour_of_day`, `suspect_check_count`, `off_hours_count`, `suspicious_togo_count`, `avg_check_duration_minutes` | One row per location × hour-of-day; temporal fraud patterns using the silver `open_hour` column. Clustered by `LocationId`. |
| `workspace.gold.gold_suspect_check_details` | `silver.suspect_fraud_checks` | all 35 silver columns, 1:1 | Receipt-level audit dataset — every column of the silver suspect checks, exposed in the gold schema. |


## Deliverable Logic (legacy → declarative)

### Table 1 — `gold_employee_fraud_scorecard`
1. **Explode** the `Discounts` array of each suspect check to attribute it to the employee who applied a discount (`d.EmployeeId` non-null)
2. **Deduplicate** on (`employee_id`, `LocationId`, `Id`) — an employee who applied several discounts on the same check is counted once
3. **Left join** `silver.employees` on `Id` for the human-readable name
4. **Group by** (`employee_id`, `employee_name`, `LocationId`) and aggregate:
   `_count(Id)` → `total_suspect_checks`, the five fraud-flag sums (boolean cast to int),
   and `_sum(total_discount_amount / total_void_amount)` → dollars at risk

### Table 2 — `gold_location_loss_prevention_summary`
* Left join `silver.revenue_centers` on `RevenueCenterId` for `revenue_center_name`
* Group by (`LocationId`, `revenue_center_name`) → `total_flagged_checks`, `total_discount_loss`,
  `total_void_loss`, and `total_dollars_at_risk` (discounts + voids)

### Table 3 — `gold_hourly_anomaly_trends`
* Group by (`LocationId`, `open_hour` as `hour_of_day`) → `suspect_check_count`,
  `off_hours_count` (irregular-hours flag, 2–7 AM), `suspicious_togo_count`,
  and `avg_check_duration_minutes` for context

### Table 4 — `gold_suspect_check_details`
* Exposes **every column** of `silver.suspect_fraud_checks` 1:1 under the gold schema.
  The legacy notebook used `CREATE OR REPLACE VIEW`; persistent UC views are not available in
  Python pipelines, so this is a materialized view instead — same columns, same 83,368 rows.

### How the imperative legacy logic became declarative

| Legacy notebook | Declarative pipeline |
| --- | --- |
| `CREATE OR REPLACE TABLE ... CLUSTER BY (LocationId) AS SELECT * FROM <temp view>` | `@dp.materialized_view(name=..., cluster_by=["LocationId"])` |
| `CREATE OR REPLACE VIEW gold_suspect_check_details` | `@dp.materialized_view` (Python pipelines cannot create persistent UC views) |
| `.orderBy(...)` before each CTAS (display-only) | dropped — row ordering is a presentation concern, applied in queries/dashboards |
| `CREATE SCHEMA IF NOT EXISTS workspace.gold_schema` | `workspace.gold` schema pre-existing; pipeline tables are auto-created on first update |
| Manual `.count()` verification prints | verification cells in this notebook + the pipeline's own update events |


## Known Results & Data-Coverage Notes (as of 2026-11-18, update `31d2fd39`)

| Deliverable | Rows | Meaning |
| --- | --- | --- |
| `gold_employee_fraud_scorecard` | 2,373 | employee × location groups; `Σ total_suspect_checks` = 59,442 employee-attributed suspect checks |
| `gold_location_loss_prevention_summary` | 127 | location × revenue-center groups |
| `gold_hourly_anomaly_trends` | 1,931 | location × hour-of-day groups |
| `gold_suspect_check_details` | 83,368 | = every row of `silver.suspect_fraud_checks` |

**Cross-checks (all verified):** `Σ total_flagged_checks` (T2) = 83,368 and `Σ suspect_check_count` (T3) = 83,368 —
every silver suspect check is counted exactly once in both summaries, matching
`silver.suspect_fraud_checks` (83,368 rows).

**Dimension names are currently NULL:** `employee_name` (T1) and `revenue_center_name` (T2) are
NULL on every row because `silver.employees` and `silver.revenue_centers` are empty (the
`rawlanding` dimension folders have no files yet). The left joins are already in place — the names
populate automatically once dimension data lands in bronze → silver.

**Coverage:** all gold counts follow directly from the silver coverage (95 days of checks,
2026-04-28 → 2026-07-31). The legacy `gold_schema` tables were built from the broader 114-day
legacy source, so absolute counts differ; the logic is identical.


In [0]:
# ============================================================================
# GOLD LAYER VERIFICATION
# ============================================================================
# Queries live tables to verify all 4 deliverables and cross-checks them
# against the silver source (expected values from the 2026-11-18 run).
# ============================================================================
from pyspark.sql.functions import sum as _sum

GOLD = "workspace.gold"
expected_counts = {
    "gold_employee_fraud_scorecard":        2_373,   # employee x location groups
    "gold_location_loss_prevention_summary":  127,   # location x revenue-center groups
    "gold_hourly_anomaly_trends":           1_931,   # location x hour-of-day groups
    "gold_suspect_check_details":          83_368,   # = silver.suspect_fraud_checks
}

print("=" * 72)
print("  Gold layer deliverables — workspace.gold")
print("=" * 72)
for name, expected in expected_counts.items():
    cnt = spark.table(f"{GOLD}.{name}").count()
    status = "\u2705 OK" if cnt == expected else f"\u274c expected {expected:,}"
    print(f"  {name:<40}{cnt:>10,}  {status}")
print("=" * 72)

# Cross-check: every silver suspect check is counted exactly once by T2 and T3
n_suspect = spark.table("workspace.silver.suspect_fraud_checks").count()
sum_t1 = spark.table(f"{GOLD}.gold_employee_fraud_scorecard").agg(_sum("total_suspect_checks").alias("s")).collect()[0]["s"]
sum_t2 = spark.table(f"{GOLD}.gold_location_loss_prevention_summary").agg(_sum("total_flagged_checks").alias("s")).collect()[0]["s"]
sum_t3 = spark.table(f"{GOLD}.gold_hourly_anomaly_trends").agg(_sum("suspect_check_count").alias("s")).collect()[0]["s"]
print(f"  silver suspect checks:                     {n_suspect:>10,}")
print(f"  Σ total_flagged_checks (T2):              {sum_t2:>10,}  {'\u2705' if sum_t2 == n_suspect else '\u274c'}")
print(f"  Σ suspect_check_count (T3):                {sum_t3:>10,}  {'\u2705' if sum_t3 == n_suspect else '\u274c'}")
print(f"  Σ total_suspect_checks (T1, attributed):   {sum_t1:>10,}  (employee-attributed subset of suspect checks)")


In [0]:
# ============================================================================
# PREVIEW: All Gold deliverables (10 rows each)
# ============================================================================
from pyspark.sql.functions import col

print("── gold_employee_fraud_scorecard (top 10 by suspect checks) ──")
display(spark.table(f"{GOLD}.gold_employee_fraud_scorecard")
        .orderBy(col("total_suspect_checks").desc())
        .limit(10))

print("\n── gold_location_loss_prevention_summary (top 10 by $ at risk) ──")
display(spark.table(f"{GOLD}.gold_location_loss_prevention_summary")
        .orderBy(col("total_dollars_at_risk").desc())
        .limit(10))

print("\n── gold_hourly_anomaly_trends (first 10 by location + hour) ──")
display(spark.table(f"{GOLD}.gold_hourly_anomaly_trends")
        .orderBy(col("LocationId").asc(), col("hour_of_day").asc())
        .limit(10))


## Running the pipeline

* Pipeline `beestats_medallion_pipeline` publishes bronze + silver + gold from three library
  files: `bronze_ingestion.py`, `silver_transformations.py`, and `gold_aggregations.py`.
  **This notebook is not a library** and is never executed by the pipeline.
* Do a **dry run** first to validate the code, then start a normal **update**. The gold
  materialized views recompute from silver whenever their inputs change — the employee
  scorecard's explode + dedup + group-by and the percentile-free aggregations are all handled
  by the pipeline's incremental MV maintenance.
* The gold tables refresh in dependency order: bronze → silver → gold, automatically.
* Once dimension data lands in `rawlanding` (employees / revenue_centers), the names in
  Tables 1 and 2 populate on the next update — no code change needed.

## Where to go next

* **Fraud Analytics Dashboard** (`Applebees Fraud Analytics Dashboard.lvdash.json`, same
  folder) — queries the gold tables; the silver/gold columns it uses are unchanged from legacy.
* The silver documentation lives at `src/transformations/silver/Applebees_Pipeline_silver_documentation`.
